<a href="https://colab.research.google.com/github/ArtyomShabunin/SMOPA-26/blob/main/lesson_10.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

<img src="https://prana-system.com/files/110/rds_color_full.png" alt="tot image" width="300"  align="center"/> &nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;&nbsp;
<img src="https://mpei.ru/AboutUniverse/OficialInfo/Attributes/PublishingImages/logo1.jpg" alt="mpei image" width="200" align="center"/>
<img src="https://mpei.ru/Structure/Universe/tanpe/structure/tfhe/PublishingImages/tot.png" alt="tot image" width="100"  align="center"/>

---

# **Системы машинного обучения и предиктивной аналитики в тепловой и возобновляемой энергетике**  

# ***Практические занятия***


---

# Занятие №10
# XGBoost
**22 апреля 2025г.**

XGBoost (Extreme Gradient Boosting) — это мощная и эффективная библиотека машинного обучения, основанная на методе **градиентного бустинга**. Она используется для решения задач классификации, регрессии и ранжирования.  

**Градиентный бустинг** — это техника **машинного обучения** для построения прогностических моделей. Она относится к классу **ансамблевых методов**, где несколько «слабых» моделей (обычно деревьев решений) объединяются в одну «сильную» модель.

Основная идея — строить модели **последовательно**, каждая из которых пытается **исправить ошибки** предыдущих, используя **градиентный спуск** для минимизации функции потерь.

### Основные особенности XGBoost:
- **Градиентный бустинг** — это метод ансамблирования, при котором несколько слабых моделей (обычно деревьев решений) объединяются для формирования сильной модели.
- **Эффективность** — XGBoost оптимизирован по скорости и использует продвинутые техники, такие как параллельные вычисления и буферизацию деревьев.
- **Поддержка регуляризации** — встроенная L1 и L2 регуляризация помогает предотвращать переобучение.
- **Обработка пропущенных значений** — библиотека умеет сама "угадывать", куда направить пропущенные значения при обучении деревьев.
- **Гибкость** — поддерживает различные типы задач, множество гиперпараметров, и может работать как с небольшими, так и с очень большими наборами данных.

### Где используется:
- Соревнования по машинному обучению (например, Kaggle)
- Финансовый анализ
- Диагностика в медицине
- Предиктивное обслуживание оборудования
- Маркетинговая аналитика

Cписок популярных альтернатив XGBoost — градиентного бустинга на решающих деревьях

1. **LightGBM (Microsoft)**
   - Быстрее XGBoost на больших данных.
   - Использует histogram-based алгоритм и leaf-wise рост дерева.
   - Хорош для задач с большим количеством признаков и категориальными данными.

2. **CatBoost (Yandex)**
   - Хорошо работает с категориальными признаками без необходимости one-hot encoding.
   - Снижает переобучение.
   - Прост в использовании: меньше параметров для настройки.

3. **HistGradientBoosting (sklearn)**
   - Встроенная реализация бустинга с гистограммами в Scikit-learn (`sklearn.ensemble.HistGradientBoostingClassifier/Regressor`).
   - Быстрый и поддерживает natively missing values.
   - Хорошая альтернатива без установки сторонних библиотек.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from tqdm import tqdm

from sklearn.model_selection import train_test_split, RandomizedSearchCV
from sklearn.metrics import (
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score,
    accuracy_score,
    classification_report,
    ConfusionMatrixDisplay,
    mean_absolute_error,
    mean_squared_error,
)
from sklearn.preprocessing import LabelEncoder

from imblearn.pipeline import Pipeline
from imblearn.under_sampling import RandomUnderSampler
from imblearn.over_sampling import SMOTE

import xgboost as xgb


## Многоклассовая классификация с XGBoost
### Загрузка и предобработка данных

In [ ]:
# import gdown
# import warnings
# warnings.filterwarnings('ignore')
# gdown.download('https://drive.google.com/uc?id=1j54o4pHTm3HvaYTEtv_i4hOJGy5yNeZZ', verify=False)

data = pd.read_parquet("./data_modes.gzip")

In [ ]:
data.head()

Признаки

In [ ]:
use_columns = ['GTA1.DBinPU.Alzzo', 'GTA1.DBinPU.Bo', 'GTA1.DBinPU.DlPkf',
               'GTA1.DBinPU.DlPtgft', 'GTA1.DBinPU.DlPvf', 'GTA1.DBinPU.fi',
               'GTA1.DBinPU.hmGTD', 'GTA1.DBinPU.hmTG', 'GTA1.DBinPU.P1mvhTG',
               'GTA1.DBinPU.Pk', 'GTA1.DBinPU.Pmvh', 'GTA1.DBinPU.PmvhMOGTD',
               'GTA1.DBinPU.PmvhMOTG', 'GTA1.DBinPU.PmvyhMOGTD',
               'GTA1.DBinPU.PmvyhMOTG', 'GTA1.DBinPU.Prazrjag_navhode',
               'GTA1.DBinPU.Ptgpd', 'GTA1.DBinPU.Ptgvh', 'GTA1.DBinPU.Pvh',
               'GTA1.DBinPU.Pvyhlg', 'GTA1.DBinPU.Qtg', 'GTA1.DBinPU.Tk',
               'GTA1.DBinPU.Tn', 'GTA1.DBinPU.Tt', 'GTA1.DBinPU.Tvh1',
               'GTA1.DBinPU.Pzad']

X = data.loc[:,use_columns]

Целевая переменная

In [ ]:
data['target'] = data[[
    'full_power_mode', 'partial_power_mode',
    'increas_power_mode', 'decreas_power_mode', 'start_up_mode',
    'shutdown_mode', 'stopped_state_mode']].idxmax(axis=1)
y = data.loc[:, ['target']]

In [ ]:
y.value_counts()

Разделение на тестовую и тренировочную выборки

In [ ]:
from sklearn.model_selection import train_test_split
# Разделяем с учетом дисбаланса классов
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42)

In [ ]:
y_train.value_counts()

In [ ]:
y_test.value_counts()

Балансировка данных (только train)


In [ ]:
# !python -m pip install imbalanced-learn xgboost


Undersampling train


In [ ]:
y_train_raw = y_train.squeeze()
y_test_eval = y_test.squeeze().copy()

sampling_strategy = {
    "full_power_mode": 1000,
    "stopped_state_mode": 1000,
    "partial_power_mode": 1000,
}
rus = RandomUnderSampler(sampling_strategy=sampling_strategy, random_state=42)
X_train_resampled, y_train_resampled = rus.fit_resample(X_train, y_train_raw)


In [ ]:
pd.Series(y_train_resampled, name='target').value_counts()


In [ ]:
# Тестовую выборку НЕ ресемплируем, чтобы метрики отражали реальное распределение классов
y_test_eval.value_counts()


In [ ]:
X_test_eval = X_test.copy()
X_test_eval.shape, y_test_eval.shape


Oversampling train


In [ ]:
X_train_resampled, y_train_resampled = SMOTE(random_state=42).fit_resample(
    X_train_resampled, y_train_resampled
)


In [ ]:
pd.Series(y_train_resampled, name='target').value_counts()


XGBoost — это алгоритм на базе деревьев решений. А деревья решений и бустинг на их основе **не чувствительны к масштабу признаков** (в отличие от моделей типа линейной регрессии, SVM или нейронных сетей).

In [ ]:
X_train_resampled.shape, X_test_eval.shape


LabelEncoding (fit только на train labels)


In [ ]:
encoder = LabelEncoder()
y_train_encoded = encoder.fit_transform(y_train_raw)
y_test_eval_encoded = encoder.transform(y_test_eval)


In [ ]:
y_train_resampled_encoded = encoder.transform(y_train_resampled)


In [ ]:
y_test_eval_encoded[:10]

In [ ]:
encoder.classes_


### Инициализация и обучение модели

In [ ]:
# xgboost импортирован в первом кодовом блоке


`xgb.XGBClassifier` — это класс из библиотеки **XGBoost**, предназначенный для решения **задач классификации** с помощью алгоритма **градиентного бустинга на деревьях решений**.

- строит ансамбль деревьев решений (по умолчанию — деревья классификации),
- на каждом шаге добавляет дерево, которое исправляет ошибки предыдущих,
- минимизирует выбранную **функцию потерь (objective)**.

---

**Применяется для:**

- **Бинарной классификации** (например, "да/нет", "спам/не спам")
- **Многоклассовой классификации** (например, классификация цифр, видов растений и т.д.)

---

Параметр **`objective`** в XGBoost `XGBClassifier` определяет **функцию потерь**, которую оптимизирует модель, и напрямую зависит от типа задачи — классификация, регрессия, ранжирование и т.д.


---

**Для задач многоклассовой классификации:**

- `'multi:softmax'` — многоклассовая классификация, возвращает **номер класса** напрямую.  
  Требуется параметр `num_class`.
- `'multi:softprob'` — многоклассовая классификация, возвращает **вектор вероятностей по каждому классу**.  
  Также требует `num_class`.

---

**Описание остальных гиперпараметров:**

- **`num_class=7`**  
  Количество уникальных классов в задаче классификации. В данном случае — 7 классов.

- **`max_depth=6`**  
  Максимальная глубина каждого дерева.  
  Глубокие деревья могут моделировать сложные зависимости, но также повышают риск переобучения.

- **`learning_rate=0.1`**
  Скорость обучения, понижает вклад каждого дерева в итоговое решение.  
  Меньшие значения требуют больше деревьев (`n_estimators`) для хорошей сходимости.

- **`n_estimators=100`**  
  Количество деревьев (итераций бустинга).  
  Итоговая модель будет объединением этих 100 слабых моделей.

- **`subsample=0.8`**  
  Доля случайно выбранных объектов для обучения каждого дерева.  
  Значения <1.0 могут помочь в борьбе с переобучением.

- **`colsample_bytree=0.8`**  
  Доля случайно выбранных признаков, используемых при построении каждого дерева.  
  Также снижает риск переобучения, особенно при большом числе признаков.

- **`random_state=42`**  
  Фиксирует генератор случайных чисел, обеспечивая воспроизводимость результатов.


In [ ]:
classifier = xgb.XGBClassifier(
    objective='multi:softprob',
    num_class=len(encoder.classes_),
    max_depth=6,
    learning_rate=0.1,
    n_estimators=100,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='mlogloss',
    random_state=42,
)

classifier.fit(X_train_resampled, y_train_resampled_encoded)


In [ ]:
y_pred_base = classifier.predict(X_test_eval)


In [ ]:
y_pred_base


In [ ]:
classifier.predict_proba(X_test_eval)[:3]


In [ ]:
np.argmax(classifier.predict_proba(X_test_eval), axis=1)[:10]


### Анализ качества модели
Матрица неточностей

In [ ]:
conf_mat = confusion_matrix(y_test_eval_encoded, y_pred_base, normalize='true')
ConfusionMatrixDisplay(conf_mat, display_labels=encoder.classes_).plot()
plt.xticks(rotation=90)
plt.show()


In [ ]:
print(classification_report(y_test_eval_encoded, y_pred_base, target_names=encoder.classes_))

accuracy_classifier = {}
precision_classifier = {}
recall_classifier = {}
f1_classifier = {}

accuracy_classifier['xgb_base'] = accuracy_score(y_test_eval_encoded, y_pred_base)
precision_classifier['xgb_base'] = precision_score(y_test_eval_encoded, y_pred_base, average='macro', zero_division=np.nan)
recall_classifier['xgb_base'] = recall_score(y_test_eval_encoded, y_pred_base, average='macro', zero_division=np.nan)
f1_classifier['xgb_base'] = f1_score(y_test_eval_encoded, y_pred_base, average='macro', zero_division=np.nan)

print(f"accuracy - {accuracy_classifier['xgb_base']*100:0.2f}%")
print(f"precision - {precision_classifier['xgb_base']*100:0.2f}%")
print(f"recall - {recall_classifier['xgb_base']*100:0.2f}%")
print(f"f1 - {f1_classifier['xgb_base']*100:0.2f}%")


### Подбор значений гиперпараметров
**RandomizedSearchCV**

In [ ]:
target_class_names = ['full_power_mode', 'stopped_state_mode', 'partial_power_mode']
class_to_id = {name: idx for idx, name in enumerate(encoder.classes_)}

sampling_strategy_cv = {
    class_to_id[name]: 1000
    for name in target_class_names
    if name in class_to_id
}

xgb_pipeline = Pipeline([
    ('under', RandomUnderSampler(sampling_strategy=sampling_strategy_cv, random_state=42)),
    ('smote', SMOTE(random_state=42)),
    ('xgb', xgb.XGBClassifier(
        objective='multi:softprob',
        num_class=len(encoder.classes_),
        eval_metric='mlogloss',
        random_state=42,
    )),
])

param_grid = {
    'xgb__max_depth': [4, 6, 8],
    'xgb__learning_rate': [0.01, 0.1, 0.2],
    'xgb__n_estimators': [100, 200],
    'xgb__subsample': [0.7, 0.8, 1.0],
    'xgb__colsample_bytree': [0.7, 0.8, 1.0],
}

random_search = RandomizedSearchCV(
    estimator=xgb_pipeline,
    param_distributions=param_grid,
    n_iter=10,
    cv=5,
    scoring='f1_macro',
    n_jobs=-1,
    verbose=2,
    random_state=42,
)

random_search.fit(X_train, y_train_encoded)

print('Лучшие параметры:', random_search.best_params_)
print('Лучший CV f1_macro:', round(random_search.best_score_, 4))


In [ ]:
# Оценка модели на тестовых данных
best_classifier = random_search.best_estimator_
y_pred_tuned = best_classifier.predict(X_test_eval)


### Анализ качества модели

In [ ]:
conf_mat = confusion_matrix(y_test_eval_encoded, y_pred_tuned, normalize='true')
ConfusionMatrixDisplay(conf_mat, display_labels=encoder.classes_).plot()
plt.xticks(rotation=90)
plt.show()


In [ ]:
print(classification_report(y_test_eval_encoded, y_pred_tuned, target_names=encoder.classes_))

accuracy_classifier['xgb_tuned'] = accuracy_score(y_test_eval_encoded, y_pred_tuned)
precision_classifier['xgb_tuned'] = precision_score(y_test_eval_encoded, y_pred_tuned, average='macro', zero_division=np.nan)
recall_classifier['xgb_tuned'] = recall_score(y_test_eval_encoded, y_pred_tuned, average='macro', zero_division=np.nan)
f1_classifier['xgb_tuned'] = f1_score(y_test_eval_encoded, y_pred_tuned, average='macro', zero_division=np.nan)

pd.DataFrame({
    'accuracy': accuracy_classifier,
    'precision': precision_classifier,
    'recall': recall_classifier,
    'f1': f1_classifier,
}).T


## Прогнозирование с XGBoost

In [ ]:
# import gdown
# import warnings
# warnings.filterwarnings('ignore')
# url = "https://drive.google.com/drive/folders/1RtrAevJUYSgTbp0YUztxEBB8_VcvjgGH?usp=drive_link"
# gdown.download_folder(url, quiet=True, verify=False)

In [ ]:
import glob
parquetFileList = glob.glob(f'./option_0/*.gzip')

In [ ]:
df_list = []

for file in tqdm(parquetFileList):
    df = pd.read_parquet(file)
    df_list.append(df)

data = pd.concat(df_list, axis=0).sort_index().ffill().drop_duplicates()
data = data.dropna()

In [ ]:
import seaborn as sns
sns.set_theme(style="whitegrid")

fig, (ax1, ax2, ax3) = plt.subplots(3, 1, figsize=(10, 6), constrained_layout=True)

ax1.plot(data['GTA1.DBinPU.P'].index, data['GTA1.DBinPU.P'].values,
         linestyle='none', marker='.', markersize=1);
ax1.set_ylabel("Мощность, кВт");
ax1.set_ylim([5700,6100]);
ax1.tick_params(labelbottom=False)

ax2.plot(data['GTA1.DBinPU.Qtg'].index, data['GTA1.DBinPU.Qtg'].values,
         linestyle='none', marker='.', markersize=1);
ax2.set_ylabel("Расход топлива, м3/ч");
ax2.set_ylim([1500,2500]);
ax2.tick_params(labelbottom=False)

ax3.plot(data['GTA1.DBinPU.Tt'].index, data['GTA1.DBinPU.Tt'].values,
         linestyle='none', marker='.', markersize=1);
ax3.set_ylabel("Температура\nух. газов, $^\\circ$С");
# ax3.set_ylim([300,600]);
ax3.tick_params(rotation=45);

### Предобработка данных

Проредим и отфильтруем данные


In [ ]:
df = pd.DataFrame(data['GTA1.DBinPU.Tt'])
df = df.resample('1h').mean()
df.columns = ['Tt']
df = df[df['Tt'] > 300]
df = df.resample('1h').asfreq()

In [ ]:
df.shape

In [ ]:
df.info()

In [ ]:
df[df['Tt'].isna()]

In [ ]:
df.index.to_series().diff().max()

In [ ]:
df.loc['2023-03-05 01:00:00':'2023-03-05 08:00:00']

In [ ]:
df.plot(style='.', figsize=(10, 2), ms=3, title="Температура ух. газов", legend=False)
plt.ylabel("$^\\circ$С")
plt.show()

**Лаги** — это **значения временного ряда на предыдущих временных шагах**, которые используются как признаки (фичи) для предсказания будущих значений.

Функция для генерации лагов

In [ ]:
def create_lag_features(df, column, lags):
    """
    Создает лаги признака column в количестве lags и возвращает новый DataFrame.
    """
    df_lagged = df.copy()
    for lag in range(1, lags + 1):
        df_lagged[f"{column}_lag_{lag}"] = df_lagged[column].shift(lag)

    return df_lagged

In [ ]:
n_lags = 24
df = create_lag_features(df, 'Tt', n_lags)

In [ ]:
df.index.to_series().diff().max()

In [ ]:
df.shape

In [ ]:
df.head()

In [ ]:
df.dropna(inplace=True)

Создаем вектор признаков и вектор целевых значений


In [ ]:
y = df.loc[:, ['Tt']]
X = df.loc[:, [col for col in df.columns if 'lag' in col]]


Разделим на тренировочную и тестовую выборки (**временной порядок важно сохранить!**)

In [ ]:
train_size = int(len(df) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = y[:train_size], y[train_size:]

assert X.index.is_monotonic_increasing
assert y.index.is_monotonic_increasing
assert X.index.is_unique
assert y.index.is_unique
assert X_train.index.max() < X_test.index.min()
assert y_train['Tt'].notna().all() and y_test['Tt'].notna().all()


### Инициализация и обучение модели

In [ ]:
pred_model = xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
)
pred_model.fit(X_train, y_train.values.ravel())


### Предсказание и базовые ориентиры

В этой ячейке считаются прогнозы основной модели XGBoost на тестовой выборке (y_pred), а также два простых baseline-прогноза для сравнения качества:

- baseline_mean_pred — среднее значение признаков в строке теста.
- baseline_lag1_pred — значение лагового признака Tt_lag_1 (наивный прогноз "как в предыдущий момент").

Далее эти три варианта будут сопоставляться по метрикам ошибки.


In [ ]:
y_pred = pred_model.predict(X_test)

baseline_mean_pred = X_test.mean(axis=1).values
baseline_lag1_pred = X_test['Tt_lag_1'].values


### Анализ качества модели

In [ ]:
def mape_nonzero(y_true, y_pred, eps=1e-6):
    mask = np.abs(y_true) > eps
    if mask.sum() == 0:
        return np.nan, 100.0
    return np.mean(np.abs((y_true[mask] - y_pred[mask]) / y_true[mask])) * 100, 100 * (1 - mask.mean())


def smape(y_true, y_pred, eps=1e-6):
    return np.mean(2.0 * np.abs(y_pred - y_true) / (np.abs(y_true) + np.abs(y_pred) + eps)) * 100


def wape(y_true, y_pred, eps=1e-6):
    return (np.abs(y_true - y_pred).sum() / (np.abs(y_true).sum() + eps)) * 100


def evaluate_regression(y_true, y_pred):
    mape_val, excl = mape_nonzero(y_true, y_pred)
    return {
        'MAE': mean_absolute_error(y_true, y_pred),
        'RMSE': np.sqrt(mean_squared_error(y_true, y_pred)),
        'MAPE': mape_val,
        'sMAPE': smape(y_true, y_pred),
        'WAPE': wape(y_true, y_pred),
        'MAPE_excluded_%': excl,
    }


y_true = y_test['Tt'].values

metrics_one_step = pd.DataFrame([
    {'model': 'xgb', **evaluate_regression(y_true, y_pred)},
    {'model': 'baseline_mean', **evaluate_regression(y_true, baseline_mean_pred)},
    {'model': 'baseline_lag1', **evaluate_regression(y_true, baseline_lag1_pred)},
]).set_index('model')

metrics_one_step


In [ ]:
plt.figure(figsize=(12, 4))
plt.plot(y_test.index, y_test['Tt'].values, label='Истина', linewidth=1)
plt.plot(y_test.index, y_pred, label='Прогноз xgb', linewidth=1)
plt.plot(y_test.index, baseline_mean_pred, label='Baseline mean', alpha=0.8)
plt.plot(y_test.index, baseline_lag1_pred, label='Baseline lag1', alpha=0.8)

plt.legend()
plt.title('Прогноз временного ряда XGBoost (one-step)')
plt.ylabel('$^\circ$С')
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()


### Автономный авторегрессионный прогноз (одна точка за раз)
На вход модели подаются предсказания, а не истинные значения из данных

In [ ]:
start = X_test.index[0]
end = X_test.index[-1]
timestamps = pd.date_range(start=start, end=end, freq='1h')


In [ ]:
predictions = []
lags = X_test.iloc[0].values.tolist()  # первые лаги из начала теста

for _ in timestamps:
    y_hat = pred_model.predict(np.array([lags]))[0]
    predictions.append(y_hat)
    lags = lags[1:] + [y_hat]

predictions = pd.DataFrame(predictions, index=timestamps, columns=['Tt'])

baseline_predictions = [X_test.iloc[0].mean() for _ in y_test.index]


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(timestamps, predictions['Tt'], label='Прогноз xgb')
plt.plot(df['Tt'].loc[start:].index, df['Tt'].loc[start:], linestyle='none', marker='.', markersize=4, label='Истина')
plt.plot(y_test.index, baseline_predictions, label='Baseline mean')

plt.ylabel('$^\circ$С')
plt.title('Autoregressive прогноз XGBoost (one-step recursive)')
plt.xlabel('Индекс времени')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.xticks(rotation=45)
plt.show()


In [ ]:
# Общие индексы
common_index = y_test.index.intersection(predictions.index)

In [ ]:
common_index = y_test.index.intersection(predictions.index)

y_true_ar = y_test.loc[common_index, 'Tt'].values
y_pred_ar = predictions.loc[common_index, 'Tt'].values
baseline_ar = np.array(baseline_predictions)[:len(common_index)]

pd.DataFrame([
    {'model': 'xgb_recursive', **evaluate_regression(y_true_ar, y_pred_ar)},
    {'model': 'baseline_mean_recursive', **evaluate_regression(y_true_ar, baseline_ar)},
]).set_index('model')


### Модель с множественным прогнозом (сразу n точек)

**sequence-to-sequence (seq2seq)** прогноз с XGBoost.
Модель получает на вход `input_len` прошлых значений и предсказывает сразу `output_len` будущих.

XGBoost **не поддерживает** напрямую многомерный выход, поэтому используем `MultiOutputRegressor`.


In [ ]:
def make_seq2seq_dataset(series, input_len, output_len):
    X, Y = [], []
    for i in range(len(series) - input_len - output_len + 1):
        x_temp = series[i:i + input_len]
        y_temp = series[i + input_len:i + input_len + output_len]

        if x_temp.isna().any() or y_temp.isna().any():
            continue

        X.append(x_temp.values)
        Y.append(y_temp.values)

    return np.array(X), np.array(Y)


In [ ]:
input_len = 20
output_len = 20
X, Y = make_seq2seq_dataset(
    df.resample('1h').asfreq()['Tt'],
    input_len=input_len, output_len=output_len)

In [ ]:
X.shape, Y.shape

In [ ]:
X[0], Y[0]

Разделение на тестовую и тренировочную выборки

In [ ]:
train_size = int(len(Y) * 0.8)
X_train, X_test = X[:train_size], X[train_size:]
y_train, y_test = Y[:train_size], Y[train_size:]

`MultiOutputRegressor`

Это обёртка, которая позволяет использовать обычную регрессионную модель
(например, `XGBRegressor`) для задач многомерной регрессии.


In [ ]:
from sklearn.multioutput import MultiOutputRegressor

model = MultiOutputRegressor(xgb.XGBRegressor(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    random_state=42,
))
model.fit(X_train, y_train)


In [ ]:
y_pred = model.predict(X_test)

In [ ]:
baseline_pred = np.array([np.ones(output_len) * x.mean() for x in X_test])


In [ ]:
sample = 0

plt.plot(range(output_len), y_test[sample], label="Истинные значения")
plt.plot(range(output_len), y_pred[sample], label="Прогноз xgb", linestyle="--")
plt.plot(range(output_len), baseline_pred[sample], label='Прогноз baseline')
plt.legend()
plt.title(f"Прогноз на {output_len} шагов вперёд")
plt.show()

In [ ]:
# np.sqrt(mean_squared_error(y_test[0], y_pred[0]))

In [ ]:
# np.sqrt(mean_squared_error(y_test[0], baseline_pred[0]))

In [ ]:
y_true_seq2seq = y_test.reshape(-1)
y_pred_seq2seq = y_pred.reshape(-1)
baseline_seq2seq = baseline_pred.reshape(-1)

pd.DataFrame([
    {'model': 'xgb_seq2seq', **evaluate_regression(y_true_seq2seq, y_pred_seq2seq)},
    {'model': 'baseline_mean_seq2seq', **evaluate_regression(y_true_seq2seq, baseline_seq2seq)},
]).set_index('model')


### Автономный авторегрессионный прогноз с seq2seq моделью


In [ ]:
# Индекс начала первого тестового горизонта в исходном ряде
start_idx = train_size + input_len
start = df.index[start_idx]
timestamps = df.index[start_idx:]


In [ ]:
current_input = X_test[0].copy()
predictions = []

# Число итераций так, чтобы покрыть весь хвост ряда
n_iters = int(np.ceil(len(timestamps) / output_len))


In [ ]:
for _ in range(n_iters):
    pred = model.predict(np.array([current_input]))[0]
    predictions.extend(pred.tolist())
    current_input = np.concatenate([current_input, pred])[-input_len:]


In [ ]:
predictions = predictions[:len(timestamps)]
predictions = pd.DataFrame(predictions, index=timestamps, columns=['Tt'])


In [ ]:
common_index = df.index.intersection(predictions.index)


In [ ]:
baseline_predictions = [X_test[0].mean() for _ in common_index]


In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(predictions.index, predictions['Tt'], label='Прогноз xgb')
plt.plot(
    df['Tt'].loc[common_index].index,
    df['Tt'].loc[common_index].values,
    linestyle='none',
    marker='.',
    markersize=4,
    label='Истинные значения',
)
plt.plot(df['Tt'].loc[common_index].index, baseline_predictions, label='Baseline mean')

plt.ylabel('$^\circ$С')
plt.title('Autoregressive прогноз XGBoost (seq2seq recursive)')
plt.xlabel('Индекс времени')
plt.legend()
plt.grid(True)
plt.tight_layout()
plt.xticks(rotation=45)
plt.show()


In [ ]:
y_true_seq2seq_recursive = df.loc[common_index, 'Tt'].values
y_pred_seq2seq_recursive = predictions.loc[common_index, 'Tt'].values
baseline_seq2seq_recursive = np.array(baseline_predictions)

pd.DataFrame([
    {'model': 'xgb_seq2seq_recursive', **evaluate_regression(y_true_seq2seq_recursive, y_pred_seq2seq_recursive)},
    {'model': 'baseline_mean_seq2seq_recursive', **evaluate_regression(y_true_seq2seq_recursive, baseline_seq2seq_recursive)},
]).set_index('model')
